<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 03. Creación de Características: Dale Pistas Nuevas a tu Modelo
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 06
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/06%20-%20Feature%20Engineering/Para%20Dummies/03_Creacion_de_Caracteristicas_Feature_Engineering_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del módulo 03 de Feature Engineering. En el cuaderno anterior de esta serie vimos cómo codificar categorías con cuidado (Target Encoding suavizado). Ahora vamos a aprender algo distinto: **crear columnas nuevas** a partir de las que ya tenemos, para ponerle a nuestro modelo la información "masticada" y más fácil de digerir.

Al terminar podrás explicar, con tus propias palabras:
1. Por qué a veces una simple operación matemática (una razón, un cuadrado, un logaritmo) mejora muchísimo un modelo.
2. Cómo crear columnas que "cuentan" cosas (por ejemplo, cuántas alertas tiene un pedido).
3. Cómo dividir y combinar columnas de texto.
4. Cómo calcular promedios "por grupo" (por ciudad, por categoría) sin hacer trampa con los datos.

---
## 1. La idea central: dale a tu modelo la pista ya masticada 🍽️

Piensa en la **sensación térmica** que anuncian en el clima. No es un instrumento que "mide sensación" directamente: es una fórmula que combina la temperatura, la humedad y el viento en **un solo número** que responde exactamente a la pregunta que nos interesa: *¿cómo se siente afuera?*

Hacer *Feature Engineering* es lo mismo: tomamos los datos que ya tenemos y los combinamos para crear una columna nueva que le facilite el trabajo al modelo. Un ejemplo clásico: si quieres predecir el **precio de un terreno cuadrado** a partir de la **longitud de su lado**, un modelo lineal lo hará muy mal, porque el precio depende del **área** ($longitud^2$), no de la longitud directamente. Pero si tú mismo calculas la columna `area = longitud ** 2` y se la das al modelo, de repente la relación con el precio **sí** es una línea recta, y el modelo lineal la aprende sin problema.

> 📌 **Para recordar:** una transformación que tú le aplicas a una columna se convierte, en la práctica, en parte del "cerebro" del modelo. Si el modelo no puede aprender una relación complicada, a veces basta con dársela ya resuelta.

In [ ]:
import pandas as pd
import numpy as np

# Un mini-dataset de terrenos cuadrados: solo conocemos la longitud del lado
terrenos = pd.DataFrame({
    "longitud_lado_m": [4, 6, 8, 10, 12, 14]
})

# El precio real depende del AREA, no de la longitud (precio = 5 millones por m2)
terrenos["precio_millones"] = (terrenos["longitud_lado_m"] ** 2) * 5

# Creamos la columna de area, que es la pista "ya masticada"
terrenos["area_m2"] = terrenos["longitud_lado_m"] ** 2

terrenos

### 🤔 ¿Qué acaba de pasar?

- Empezamos importando `pandas` y `numpy`, las herramientas base de este cuaderno.
- `terrenos["longitud_lado_m"] ** 2` calcula el área de cada terreno, elevando al cuadrado la longitud de su lado.
- Fíjate que `precio_millones` es exactamente `area_m2 * 5`: una relación **perfectamente lineal** con el área, pero **no lineal** con la longitud.
- Si entrenáramos un modelo lineal solo con `longitud_lado_m`, se equivocaría bastante. Si le damos `area_m2`, aprendería la relación exacta. Esa es toda la magia de crear una buena característica: no le damos información nueva de la nada, le damos la **misma** información en una forma que le resulta más fácil de usar.

---
## 2. Transformaciones matemáticas: razones y logaritmos 🧮

Dos trucos matemáticos muy usados:

- **Razones (ratios):** dividir una columna entre otra suele revelar "eficiencia" — por ejemplo, litros de gasolina consumidos por cada 100 km, o precio por metro cuadrado.
- **Logaritmos:** cuando una columna tiene una distribución muy "desigual" (unos pocos valores enormes y el resto pequeños, como los ingresos o el patrimonio), aplicar $\log(1+x)$ la vuelve mucho más "pareja" y fácil de trabajar para muchos modelos.

In [ ]:
# Ejemplo de razon: relacion peso/potencia en carros (entre mas bajo, mas agil)
carros = pd.DataFrame({
    "modelo": ["Compacto", "Sedan", "Deportivo", "Camioneta"],
    "peso_kg": [1100, 1400, 1300, 2200],
    "potencia_hp": [90, 120, 280, 150]
})
carros["peso_por_hp"] = carros["peso_kg"] / carros["potencia_hp"]

# Ejemplo de logaritmo: ingresos con un valor extremo (un caso muy rico)
ingresos = pd.DataFrame({
    "persona": ["A", "B", "C", "D", "E"],
    "ingreso_mensual": [1200, 1500, 1800, 2000, 90000]
})
ingresos["log_ingreso"] = np.log1p(ingresos["ingreso_mensual"])

print(carros)
print()
print(ingresos)

### 🤔 ¿Qué acaba de pasar?

- `peso_kg / potencia_hp` crea una razón: el **Deportivo** tiene el valor más bajo (4.6), confirmando que es el más ágil por caballo de fuerza, aunque no sea el más liviano.
- `np.log1p(x)` calcula $\log(1+x)$ en vez de $\log(x)$; usamos la versión "+1" porque así funciona incluso si algún valor fuera 0 (el logaritmo de 0 no existe).
- Compara `ingreso_mensual` con `log_ingreso`: en la columna original, la persona "E" está kilómetros de distancia del resto (90,000 contra ~1,500). En la columna logarítmica, esa diferencia se "encoge" muchísimo (11.4 contra ~7.3), lo que ayuda a que un modelo no se obsesione con ese único caso extremo.

---
## 3. Conteos: convertir "presencia/ausencia" en un número 🔢

Muchas veces tenemos varias columnas de Sí/No (o `True`/`False`) que en el fondo describen "cuántas cosas de cierto tipo están presentes". En Python, `True` vale 1 y `False` vale 0, así que **sumar** columnas booleanas nos da automáticamente un conteo.

Piensa en un pedido de comida a domicilio: ¿tiene extra queso? ¿tiene bebida? ¿tiene postre? Sumar esas respuestas nos da un solo número: "cuántos extras tiene el pedido".

In [ ]:
pedidos = pd.DataFrame({
    "pedido_id": [1, 2, 3, 4],
    "extra_queso": [True, False, True, True],
    "bebida":      [True, True, False, True],
    "postre":      [False, False, True, True],
})

extras = ["extra_queso", "bebida", "postre"]

# Sumamos a lo largo de las columnas (axis=1) para contar cuantos "Si" tiene cada pedido
pedidos["cantidad_extras"] = pedidos[extras].sum(axis=1)
pedidos

### 🤔 ¿Qué acaba de pasar?

- `pedidos[extras]` selecciona solo las tres columnas booleanas.
- `.sum(axis=1)` suma **a lo largo de las columnas**, es decir, fila por fila (por eso `axis=1` y no `axis=0`, que sumaría hacia abajo).
- El pedido 4 tiene `cantidad_extras = 3` porque las tres columnas eran `True`. El pedido 3 tiene `2`.
- Esta idea de "sumar banderas booleanas" es exactamente la misma que se usa, por ejemplo, para contar cuántos elementos de riesgo (semáforo, cruce peatonal, reductor de velocidad) había cerca de un accidente de tránsito: una sola columna nueva resume varias columnas de Sí/No.

---
## 4. Dividir y combinar columnas de texto 🔤

A veces el texto que tenemos en una columna en realidad **junta dos datos en uno**, y nos conviene separarlos. Otras veces es al revés: nos conviene **unir** dos columnas para capturar una combinación interesante que por separado no se nota.

Ejemplo típico: una columna `"Plan Nivel"` como `"Corporativo L3"`, donde en realidad hay dos datos escondidos: el tipo de plan y el nivel.

In [ ]:
clientes = pd.DataFrame({
    "cliente": ["Ana", "Luis", "Marta", "Carlos"],
    "plan": ["Corporativo L3", "Personal L1", "Personal L3", "Corporativo L2"],
    "ciudad": ["Tunja", "Duitama", "Tunja", "Sogamoso"],
    "estado_laboral": ["Empleado", "Independiente", "Empleado", "Empleado"],
})

# Dividimos "plan" en dos columnas nuevas, separando por el espacio en blanco
clientes[["tipo_plan", "nivel_plan"]] = clientes["plan"].str.split(" ", expand=True)

# Combinamos dos columnas categoricas en una sola (interaccion)
clientes["ciudad_y_estado"] = clientes["ciudad"] + "_" + clientes["estado_laboral"]

clientes

### 🤔 ¿Qué acaba de pasar?

- `.str.split(" ", expand=True)` es el accesor `.str`, que nos deja aplicar operaciones de texto columna por columna. Con `expand=True`, el resultado de dividir cada texto se reparte en columnas nuevas en lugar de quedar como una lista dentro de una sola celda.
- Ahora `tipo_plan` (`"Corporativo"`, `"Personal"`) y `nivel_plan` (`"L3"`, `"L1"`, `"L2"`) son columnas independientes que el modelo puede usar por separado.
- `clientes["ciudad"] + "_" + clientes["estado_laboral"]` concatena texto con texto, creando una categoría combinada como `"Tunja_Empleado"`. Esto es útil cuando sospechas que la combinación de dos categorías dice algo que ninguna de las dos dice por separado (por ejemplo, tal vez "Sogamoso + Independiente" tiene un comportamiento particular que no se ve mirando "Sogamoso" o "Independiente" por separado).

---
## 5. Transformaciones agrupadas: el promedio "por grupo" 📊

Otra columna muy poderosa: **"el promedio de tu grupo"**. Por ejemplo, "el ingreso promedio de tu ciudad" o "el precio promedio de tu categoría de producto". En Pandas esto se hace con `.groupby(...).transform(...)`.

> ⚠️ **Cuidado con la trampa:** si vas a separar tus datos en un conjunto de **entrenamiento** y uno de **validación**, el promedio por grupo debe calcularse **solo con los datos de entrenamiento**. Si lo calculas con todos los datos (incluida la validación) estás dejando que la validación "se entere" de información que no debería conocer todavía — a esto se le llama **fuga de datos (data leakage)**, y hace que tu modelo parezca mejor de lo que realmente es.

In [ ]:
ventas = pd.DataFrame({
    "producto_id": range(1, 9),
    "ciudad": ["Tunja", "Tunja", "Duitama", "Duitama", "Tunja", "Sogamoso", "Sogamoso", "Duitama"],
    "precio": [120, 135, 98, 105, 128, 80, 85, 110],
})

# Paso 1: dividimos en un conjunto de "entrenamiento" y uno de "validacion"
entrenamiento = ventas.sample(frac=0.6, random_state=0)
validacion = ventas.drop(entrenamiento.index)

# Paso 2: calculamos el precio promedio por ciudad SOLO en entrenamiento
entrenamiento = entrenamiento.copy()
entrenamiento["precio_promedio_ciudad"] = (
    entrenamiento.groupby("ciudad")["precio"].transform("mean")
)

# Paso 3: trasladamos ese promedio a validacion con un merge (NO lo recalculamos ahi)
tabla_promedios = entrenamiento[["ciudad", "precio_promedio_ciudad"]].drop_duplicates()
validacion = validacion.merge(tabla_promedios, on="ciudad", how="left")

print("Entrenamiento:")
print(entrenamiento[["ciudad", "precio", "precio_promedio_ciudad"]])
print()
print("Validacion (usa el promedio aprendido en entrenamiento):")
print(validacion[["ciudad", "precio", "precio_promedio_ciudad"]])

### 🤔 ¿Qué acaba de pasar?

- `.groupby("ciudad")["precio"].transform("mean")` calcula el precio promedio de cada ciudad y lo "reparte" de vuelta a cada fila que pertenece a esa ciudad (por eso usamos `transform` y no solo `mean`, que colapsaría las filas).
- Hicimos ese cálculo **solo** sobre `entrenamiento`, nunca sobre `validacion`.
- Para llevar esos promedios a `validacion`, usamos `.merge(..., on="ciudad", how="left")`: es como buscar en una tabla de referencia ("Tunja → 127.6", "Duitama → 101.5"...) el valor que le corresponde a cada fila, sin recalcular nada con los datos de validación.
- Si una ciudad de `validacion` no existiera en `entrenamiento`, su promedio quedaría vacío (`NaN`) — una señal honesta de "no tengo esa información todavía", en vez de un número inventado con datos que no deberíamos haber visto.

---
## 6. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| Principio general | Una transformación que tú aplicas a una columna se vuelve parte del "cerebro" del modelo. |
| Razones (ratios) | Dividir una columna entre otra revela eficiencia (ej. peso/potencia). |
| Logaritmo (`log1p`) | "Encoge" columnas con valores extremos y las vuelve más manejables. |
| Conteos | Sumar columnas booleanas (`.sum(axis=1)`) cuenta cuántas condiciones se cumplen. |
| Dividir texto | `.str.split(..., expand=True)` separa un texto en columnas nuevas. |
| Combinar texto | Concatenar columnas (`col1 + "_" + col2`) captura interacciones entre categorías. |
| Transformación agrupada | `.groupby(...).transform("mean")` crea la columna "el promedio de tu grupo". |
| Fuga de datos | El promedio por grupo se calcula solo en entrenamiento, y se traslada a validación con `.merge()`. |

➡️ **Siguiente paso:** en el cuaderno [04 - PCA (Para Dummies)](04_PCA_Feature_Engineering_Dummies.ipynb) aprenderás una forma distinta de crear características: en vez de combinar columnas "a mano", dejaremos que un algoritmo encuentre automáticamente las combinaciones más importantes.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>